# Nemotron Local CV and Advanced Error Dashboard

This Kaggle-oriented notebook evaluates a submitted LoRA adapter against a local validation split using scoring logic aligned with the competition metric. It then exports per-task metrics, error cases, report tables, and publication-ready figures.

The workflow automatically discovers either an extracted adapter or `submission.zip`, supports smoke tests, and produces a reusable analysis bundle under `analysis_outputs/` and `figures/`.

**Primary outputs:** local CV score, `debug_predictions.csv`, detailed error tables, dashboard figures, and a validated `submission.zip`.


In [ ]:
from pathlib import Path

print("All submission.zip:")
for p in Path("/kaggle/input").rglob("submission.zip"):
    print(p)

print("\nAll adapter_config.json:")
for p in Path("/kaggle/input").rglob("adapter_config.json"):
    print(p)

print("\nAll adapter_model.safetensors:")
for p in Path("/kaggle/input").rglob("adapter_model.safetensors"):
    print(p)

## 1. Prepare environment


In [ ]:
"""Prepare the Kaggle metric runtime used by the NVIDIA Nemotron challenge."""

import subprocess
import sys

commands = [
    'uv pip uninstall torch torchvision torchaudio',
    'tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp',
    'chmod +x /tmp/triton/backends/nvidia/bin/ptxas',
    'chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell',
]

for cmd in commands:
    print(f'Running: {cmd}')
    subprocess.run(cmd, shell=True, check=True)

sys.path.insert(0, '/tmp')


## 2. Imports and configuration


In [ ]:
import glob
import json
import math
import multiprocessing
import os
import re
import shutil
import time
import zipfile
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm import tqdm

# =========================
# User configuration
# =========================
CFG = {
    # Input discovery
    'VAL_PATH': 'AUTO',                       # 'AUTO' or a concrete path to val_set_950.jsonl
    'VAL_FILENAME': 'val_set_950.jsonl',
    'ADAPTER_DIR': 'AUTO',                    # 'AUTO' or a directory containing adapter_config.json and adapter_model.safetensors
    'SUBMISSION_ZIP': None,                   # Optional: path to submission.zip. If set, it has priority over ADAPTER_DIR.

    # Running mode
    'RUN_INFERENCE': True,                    # Set False to only analyze existing debug_predictions.csv + test.csv
    'FORCE_RERUN': True,                      # If False and debug_predictions.csv exists, skip expensive inference
    'SMOKE_TEST': False,                      # Set True for a quick run before full validation
    'SMOKE_N': 50,
    'RANDOM_SEED': 42,

    # vLLM / metric parameters
    'MAX_LORA_RANK': 32,
    'MAX_TOKENS': 7680,
    'TOP_P': 1.0,
    'TEMPERATURE': 0.0,
    'MAX_NUM_SEQS': 64,
    'GPU_MEMORY_UTILIZATION': 0.85,
    'MAX_MODEL_LEN': 8192,

    # Output directories
    'ANALYSIS_DIR': 'analysis_outputs',
    'FIGURE_DIR': 'figures',
    'PLOT_DPI': 300,                         # All exported figures use 300 DPI
    'TABLE_DIR': 'plot_tables',              # Per-figure source tables under ANALYSIS_DIR

    # Optional advanced-dashboard inputs. Leave empty unless you have these files.
    # Multi-version local-CV result files can be CSV/JSONL with either is_correct or id + extracted_prediction.
    'VERSION_RESULT_FILES': [],
    # Training/source mixture files can be CSV/JSONL with source/data_source/dataset/origin and label/category columns.
    'TRAIN_DATA_FILES': [],
    # Self-consistency result files can be CSV/JSONL with vote_1, vote_2, ... and final_vote/majority_vote columns.
    'SELF_CONSISTENCY_FILES': [],
}


ANALYSIS_DIR = Path(CFG['ANALYSIS_DIR'])
FIGURE_DIR = Path(CFG['FIGURE_DIR'])
TABLE_DIR = ANALYSIS_DIR / CFG.get('TABLE_DIR', 'plot_tables')
ANALYSIS_DIR.mkdir(exist_ok=True, parents=True)
FIGURE_DIR.mkdir(exist_ok=True, parents=True)
TABLE_DIR.mkdir(exist_ok=True, parents=True)

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_rows', 100)

print(json.dumps(CFG, indent=2))


In [ ]:
from pathlib import Path
import zipfile
import shutil

from pathlib import Path

def find_submission_zip(input_root="/kaggle/input"):
    input_root = Path(input_root)

    # 1. 优先查找 Kaggle imported notebook outputs 里的 submission.zip
    candidates = list(input_root.glob("notebooks/*/*/submission.zip"))

    # 2. 如果没找到，再全局搜索
    if not candidates:
        candidates = list(input_root.rglob("submission.zip"))

    if not candidates:
        raise FileNotFoundError(
            "No submission.zip found under /kaggle/input. "
            "Please make sure you have added the notebook output as Kaggle input."
        )

    # 3. 如果有多个，优先选择路径里带 notebooks 的
    candidates = sorted(
        candidates,
        key=lambda p: (
            "notebooks" not in str(p),
            len(str(p))
        )
    )

    print("Found submission.zip candidates:")
    for p in candidates:
        print(" ", p)

    selected = candidates[0]
    print("\nAuto-selected SUBMISSION_ZIP:", selected)

    return selected


SUBMISSION_ZIP = find_submission_zip()

assert SUBMISSION_ZIP.exists(), f"submission.zip not found: {SUBMISSION_ZIP}"

print("Using submission.zip:", SUBMISSION_ZIP)

# Check files inside zip
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as z:
    names = z.namelist()
    print("\nFiles inside submission.zip:")
    for name in names:
        print(" -", name)

    assert any(name.endswith("adapter_config.json") for name in names), \
        "adapter_config.json not found inside submission.zip"
    assert any(name.endswith("adapter_model.safetensors") for name in names), \
        "adapter_model.safetensors not found inside submission.zip"

    # Extract to temporary folder under /kaggle/working
    extract_dir = Path("/kaggle/working/extracted_submission")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    z.extractall(extract_dir)

# Find extracted adapter files
adapter_config = list(extract_dir.rglob("adapter_config.json"))[0]
adapter_model = list(extract_dir.rglob("adapter_model.safetensors"))[0]

# Copy to /kaggle/working root
shutil.copy(adapter_config, "/kaggle/working/adapter_config.json")
shutil.copy(adapter_model, "/kaggle/working/adapter_model.safetensors")

print("\nPrepared adapter files:")
print("/kaggle/working/adapter_config.json")
print("/kaggle/working/adapter_model.safetensors")

print("\nCheck:")
print("adapter_config exists:", Path("/kaggle/working/adapter_config.json").exists())
print("adapter_model exists:", Path("/kaggle/working/adapter_model.safetensors").exists())

## 3. Locate validation file and adapter


In [ ]:
def find_first_file(root: str | Path, filename: str) -> Path:
    """Return the first matching file under root, preferring shorter paths for readability."""
    root = Path(root)
    matches = sorted(root.rglob(filename), key=lambda p: (len(str(p)), str(p)))
    if not matches:
        raise FileNotFoundError(f'Cannot find {filename} under {root}')
    print(f'Found {filename}: {matches[0]}')
    return matches[0]


def find_adapter_dirs(root: str | Path = '/kaggle/input') -> list[Path]:
    """Find directories that contain both required LoRA adapter files."""
    root = Path(root)
    dirs = []
    for cfg_path in root.rglob('adapter_config.json'):
        candidate_dir = cfg_path.parent
        if (candidate_dir / 'adapter_model.safetensors').exists():
            dirs.append(candidate_dir)
    dirs = sorted(set(dirs), key=lambda p: (len(str(p)), str(p)))
    return dirs


def copy_adapter_to_working(adapter_dir: str | Path, work_dir: str | Path = '.') -> Path:
    """Copy adapter files to the working directory and return the working adapter path."""
    adapter_dir = Path(adapter_dir)
    work_dir = Path(work_dir)
    required = ['adapter_config.json', 'adapter_model.safetensors']
    for name in required:
        src = adapter_dir / name
        if not src.exists():
            raise FileNotFoundError(f'Missing {src}')
        shutil.copy2(src, work_dir / name)
    print(f'Copied adapter files from: {adapter_dir}')
    return work_dir


def unzip_submission_to_working(zip_path: str | Path, work_dir: str | Path = '.') -> Path:
    """Unzip submission.zip and locate the adapter directory."""
    zip_path = Path(zip_path)
    work_dir = Path(work_dir)
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(work_dir)
    print(f'Extracted submission zip: {zip_path}')

    local_adapter_dirs = find_adapter_dirs(work_dir)
    if not local_adapter_dirs:
        raise FileNotFoundError('No adapter files found after unzipping submission.zip')
    return copy_adapter_to_working(local_adapter_dirs[0], work_dir)


def prepare_adapter() -> Path:
    """Resolve adapter source and place adapter files in the current working directory."""
    # Priority 1: explicit submission.zip
    if CFG['SUBMISSION_ZIP']:
        return unzip_submission_to_working(CFG['SUBMISSION_ZIP'], '.')

    # Priority 2: current working directory already contains adapter files
    if Path('adapter_config.json').exists() and Path('adapter_model.safetensors').exists():
        print('Using adapter files already present in current working directory.')
        return Path('.')

    # Priority 3: explicit adapter directory
    if CFG['ADAPTER_DIR'] != 'AUTO':
        return copy_adapter_to_working(CFG['ADAPTER_DIR'], '.')

    # Priority 4: auto-discovery under /kaggle/input
    adapter_dirs = find_adapter_dirs('/kaggle/input')
    if not adapter_dirs:
        raise FileNotFoundError(
            'No adapter directory found under /kaggle/input. '
            'Attach a dataset/notebook output containing adapter_config.json and adapter_model.safetensors.'
        )

    print('Candidate adapter directories:')
    for p in adapter_dirs[:10]:
        print(' -', p)

    return copy_adapter_to_working(adapter_dirs[0], '.')


# Resolve validation file
if CFG['VAL_PATH'] == 'AUTO':
    VAL_PATH = find_first_file('/kaggle/input', CFG['VAL_FILENAME'])
else:
    VAL_PATH = Path(CFG['VAL_PATH'])
    if not VAL_PATH.exists():
        raise FileNotFoundError(VAL_PATH)

# Resolve adapter files
from pathlib import Path

ADAPTER_DIR = Path("/kaggle/working")
print("Using adapter directory:", ADAPTER_DIR)
print('Using validation file:', VAL_PATH)
print('Using adapter directory:', ADAPTER_DIR.resolve())


## 4. Load model path and validation data


In [ ]:
# Base model used by the official metric.
MODEL_PATH = kagglehub.model_download(
    'metric/nemotron-3-nano-30b-a3b-bf16/transformers/default'
)
print('MODEL_PATH:', MODEL_PATH)


In [ ]:
val_df = pd.read_json(VAL_PATH, lines=True)

required_cols = {'id', 'prompt', 'answer'}
missing_cols = required_cols - set(val_df.columns)
if missing_cols:
    raise ValueError(f'Validation file is missing required columns: {missing_cols}')

if 'label' not in val_df.columns:
    print('Warning: validation file has no label column. Per-label plots will be skipped or limited.')

if CFG['SMOKE_TEST']:
    n = min(CFG['SMOKE_N'], len(val_df))
    val_df = val_df.sample(n=n, random_state=CFG['RANDOM_SEED']).reset_index(drop=True)
    print(f'SMOKE_TEST=True: sampled {len(val_df)} validation rows.')
else:
    print(f'Full validation size: {len(val_df)}')

print('Unique ids:', val_df['id'].nunique())
display(val_df.head())

# The official metric reads DATA_PATH / test.csv.
val_df.to_csv('./test.csv', index=False, header=True)
DATA_PATH = Path('./')


## 5. Official-like metric code


In [ ]:
class ParticipantVisibleError(Exception):
    pass


def cache_model(
    path: str | Path,
    exts: tuple[str, ...] = ('.bin', '.pt', '.safetensors'),
    num_workers: int | None = None,
    chunk_mb: int = 256,
) -> int:
    """Pre-read model weight files into the OS page cache to speed up later loads.

    Args:
        path        : Directory containing model files, or a single file path.
        exts        : File extensions treated as model weight files.
        num_workers : Number of threads (default = min(CPU cores, 8)).
        chunk_mb    : Size of each read chunk in MB.

    Returns:
        Total bytes read (int).
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def warmup_file(fpath: Path) -> tuple[Path, int]:
        """Sequentially read an entire file in chunks."""
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        try:
            with open(fpath, 'rb') as f:
                while True:
                    data = f.read(chunk_size)
                    if not data:
                        break
                    total += len(data)
        except Exception as e:
            print(f'Error reading {fpath}: {e}')
        return fpath, total

    path = Path(path)
    # Collect files to read
    files: list[Path] = []
    if path.is_dir():
        files = [p for p in path.rglob('*') if p.is_file() and str(p).endswith(exts)]
        files.sort()
    else:
        files = [path] if path.exists() else []

    if not files:
        print(f'No model files found to cache at: {path}')
        return 0

    # Decide number of worker threads
    if num_workers is None:
        try:
            num_workers = min(multiprocessing.cpu_count(), 8)
        except Exception:
            num_workers = 4

    print(f'[cache_model] {len(files)} file(s), {num_workers} worker(s)')
    t0 = time.time()
    total_bytes = 0
    # Read files in parallel
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n
            print(f'[{i}/{len(files)}] cached {fpath.name}')

    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    speed = gb / elapsed if elapsed > 0 else 0
    print(f'[cache_model] total read ≈ {gb:.2f} GB')
    print(f'[cache_model] elapsed {elapsed:.2f} s, ~{speed:.2f} GB/s')
    return total_bytes


def extract_final_answer(text: str | None) -> str:
    r"""Extracts the final answer from the model response.

    Prioritizes extracting answers inside `\boxed{}`.
    If no `\boxed{}` format is found, attempts to extract numbers from other formats.

    Examples:
        >>> extract_final_answer(r"The answer is \boxed{42}")
        '42'
        >>> extract_final_answer("The final answer is: 3.14")
        '3.14'
        >>> extract_final_answer("Just a number 100 in text")
        '100'
        >>> extract_final_answer(None)
        'NOT_FOUND'
    """
    if text is None:
        return 'NOT_FOUND'

    # Search for boxed answer
    # Match all instances of \boxed{...} or unclosed \boxed{ at the end
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    # Other common formats if \boxed{} is not found
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    # If no structured format is found, extract the last valid number in the text
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]

    # If no numeric answer is found, return the last line of text as a fallback
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'


def verify(stored_answer: str, predicted: str) -> bool:
    """Verify if the answer matches.

    For numerical answers, allow them to be judged as equal within a certain relative tolerance (1e-2);
    otherwise, compare strictly as strings (case-insensitive).

    Examples:
        >>> verify("10011000", "10011000")
        True
        >>> verify("10011000", "10011001")
        False
        >>> verify("24.64", "24.6401")
        True
        >>> verify("XLVII", "xlvii")
        True
        >>> verify("11011", "00011011")
        False
    """
    # Clean up strings
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()

    # If the answer is a binary string, compare strictly as strings
    if re.fullmatch(r'[01]+', stored_answer):
        return predicted.lower() == stored_answer.lower()

    try:
        # Try to convert the answers to floating point numbers
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        # Use a small absolute tolerance for numbers near zero
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        # Fallback to case-insensitive string comparison
        return predicted.lower() == stored_answer.lower()


def generate_standard_submission(submission_dir: str):
    """Processes an extracted submission archive to produce a standard submission file."""
    # Locate the LoRA files within the extracted directory
    possible_extraction_dirs = {
        '/kaggle/tmp',
        '/kaggle/working',
        submission_dir,
    }
    adapter_configs = []
    for search_dir in possible_extraction_dirs:
        if os.path.exists(search_dir):
            adapter_configs.extend(
                glob.glob(
                    os.path.join(search_dir, '**/adapter_config.json'), recursive=True
                )
            )
    if not adapter_configs:
        raise ParticipantVisibleError(
            'No adapter_config.json found in submission. Found:\n\n'
            f'{submission_dir} {os.listdir(submission_dir)}\n\n'
            f'/kaggle/tmp {os.listdir("/kaggle/tmp")}\n\n'
            f'/kaggle/input/competition_evaluation {os.listdir("/kaggle/input/competition_evaluation")}'
        )

    lora_path = os.path.dirname(adapter_configs[0])

    # Load test data
    test_df = pd.read_csv(DATA_PATH / 'test.csv', index_col=None)

    row_id_col = str(test_df.columns.to_list()[0])
    predictions = []
    for item in test_df.itertuples(index=False):
        predictions.append(
            {
                row_id_col: getattr(item, row_id_col),
                'prediction': lora_path,
            }
        )

    submission_df = pd.DataFrame(predictions)

    # Write the standard submission file to the current working directory
    submission_df.to_csv('submission.csv', index=False)


def generate_predictions(
    test_df: pd.DataFrame,
    lora_path: str,
    row_id_col: str,
    max_lora_rank: int,
    max_tokens: int,
    top_p: float,
    temperature: float,
    max_num_seqs: int,
    gpu_memory_utilization: float,
    max_model_len: int,
    debug: bool = False,
) -> pd.DataFrame:
    """Load the model and generate predictions for the provided test data.

    Args:
        debug: If True, writes a CSV file with raw model outputs and extracted predictions.
    """
    # Cache Model
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

    os.environ['TRANSFORMERS_NO_TF'] = '1'
    os.environ['TRANSFORMERS_NO_FLAX'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    os.environ['TRITON_PTXAS_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas'

    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    # Initialize vLLM Offline inference Engine
    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype='auto',
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )

    tokenizer = llm.get_tokenizer()
    prompts = []
    for item in test_df.itertuples(index=False):
        user_content = (
            item.prompt
            + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
        )
        # Format using the tokenizer's chat template directly
        try:
            prompt = tokenizer.apply_chat_template(
                [{'role': 'user', 'content': user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            # Fallback if chat template fails
            prompt = user_content
        prompts.append(prompt)

    # Generate predictions using continuous batching
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest('adapter', 1, lora_path),
    )

    predictions = []
    debug_records = []
    for item, output in zip(test_df.itertuples(index=False), outputs):
        raw_text = output.outputs[0].text
        extracted_answer = extract_final_answer(raw_text)

        row_id_val = getattr(item, row_id_col)

        predictions.append(
            {
                row_id_col: row_id_val,
                'prediction': extracted_answer,
            }
        )

        if debug:
            debug_records.append(
                {
                    row_id_col: row_id_val,
                    'raw_output': raw_text,
                    'extracted_prediction': extracted_answer,
                }
            )

    # Write debug CSV if requested
    if debug and debug_records:
        debug_df = pd.DataFrame(debug_records)
        debug_df.to_csv('debug_predictions.csv', index=False)
        debug_df.to_json(
            'debug_predictions.jsonl',
            orient='records',
            lines=True,
            force_ascii=False,
        )
        
        print('Debug data saved to debug_predictions.csv')

    # for debug
    pd.DataFrame(predictions).to_json(
        'check_predictions.jsonl',
        orient='records',
        lines=True,
        force_ascii=False,
    )
    
    return pd.DataFrame(predictions)


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    max_lora_rank: int = 32,
    max_tokens: int = 3584,
    top_p: float = 1.0,
    temperature: float = 1.0,
    max_num_seqs: int = 128,
    gpu_memory_utilization: float = 0.85,
    max_model_len: int = 4096,
    debug: bool = False,
) -> float:
    r"""Evaluate the generated predictions against the ground truth.

    Submissions are evaluated based on their **Accuracy** in solving the provided
    tasks. The NVIDIA Nemotron-3-Nano-30B model is loaded with the participant's
    submitted LoRA adapter (which must include an `adapter_config.json`) using
    the vLLM inference engine. For each test case, the model is prompted to
    generate a response and instructed to place its final answer within a `\boxed{}`
    LaTeX command. The metric extracts the final answer from the generated text,
    prioritizing content within the boxed format while falling back to other
    heuristic patterns or the first numeric value found. A prediction is graded as
    correct if it matches the ground truth either exactly as a string or within a
    relative numerical tolerance of $10^{-2}$. The final score is the proportion of
    correctly answered questions.

    Args:
        solution: DataFrame containing the ground truth answers. Must include the
            row_id_column_name and an 'answer' column.
        submission: DataFrame containing the predicted answers. Must include the
            row_id_column_name and a 'prediction' column.
        row_id_column_name: The name of the ID column used to join solution and
            submission.
        max_lora_rank: Maximum rank for LoRA adapters.
        max_tokens: Maximum number of tokens to generate.
        top_p: Top-p sampling parameter.
        temperature: Temperature sampling parameter.
        max_num_seqs: Maximum number of sequences to process concurrently.
        gpu_memory_utilization: Fraction of GPU memory to allocate for the vLLM execution.
        max_model_len: Maximum context length (input + output tokens).
        debug: If True, writes raw outputs and extracted predictions to a CSV file.

    Returns:
        The accuracy score (fraction of matches) as a float.
    """
    lora_path = submission['prediction'].iloc[0]

    # Load test data and filter it to only include rows present in the solution
    test_df = pd.read_csv(DATA_PATH / 'test.csv', index_col=None)
    row_id_col = str(test_df.columns.to_list()[0])
    test_df = test_df[test_df[row_id_col].isin(solution[row_id_column_name])]

    submission = generate_predictions(
        test_df=test_df,
        lora_path=lora_path,
        row_id_col=row_id_column_name,
        max_lora_rank=max_lora_rank,
        max_tokens=max_tokens,
        top_p=top_p,
        temperature=temperature,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len,
        debug=debug,
    )

    dataset = solution.merge(submission, on=row_id_column_name)
    num_correct = 0

    # Verify the predictions
    for item in dataset.itertuples(index=False):
        ground_truth = item.answer
        extracted_answer = item.prediction

        match = verify(str(ground_truth), str(extracted_answer))
        if match:
            num_correct += 1

    accuracy = num_correct / len(solution)
    return float(accuracy)


## 6. Run local CV inference and scoring


In [ ]:
should_run = CFG['RUN_INFERENCE']
if Path('debug_predictions.csv').exists() and not CFG['FORCE_RERUN']:
    print('debug_predictions.csv already exists and FORCE_RERUN=False; skipping inference.')
    should_run = False

if should_run:
    val_score = score(
        solution=pd.read_csv('./test.csv'),
        submission=pd.DataFrame([{'prediction': str(ADAPTER_DIR)}]),
        row_id_column_name='id',
        max_lora_rank=CFG['MAX_LORA_RANK'],
        max_tokens=CFG['MAX_TOKENS'],
        top_p=CFG['TOP_P'],
        temperature=CFG['TEMPERATURE'],
        max_num_seqs=CFG['MAX_NUM_SEQS'],
        gpu_memory_utilization=CFG['GPU_MEMORY_UTILIZATION'],
        max_model_len=CFG['MAX_MODEL_LEN'],
        debug=True,
    )
    print(f'Local CV score: {val_score:.6f}')
else:
    val_score = None
    print('Inference skipped. The analysis section will use existing debug_predictions.csv.')


## 7. Build merged analysis table


In [ ]:
def build_merge_df(solution_path: str | Path = './test.csv', pred_path: str | Path = 'debug_predictions.csv') -> pd.DataFrame:
    """Merge ground truth and debug predictions, then compute correctness and diagnostics."""
    solution_df = pd.read_csv(solution_path, dtype='str')
    pred_df = pd.read_csv(pred_path, dtype='str')

    if 'extracted_prediction' not in pred_df.columns:
        raise ValueError('debug_predictions.csv must contain extracted_prediction')
    if 'raw_output' not in pred_df.columns:
        pred_df['raw_output'] = ''

    pred_df['extracted_prediction'] = pred_df['extracted_prediction'].fillna('NOT_FOUND')
    pred_df['raw_output'] = pred_df['raw_output'].fillna('')

    merged = pd.merge(solution_df, pred_df, how='left', on='id')
    merged['extracted_prediction'] = merged['extracted_prediction'].fillna('NOT_FOUND')
    merged['raw_output'] = merged['raw_output'].fillna('')
    merged['is_correct'] = merged.apply(
        lambda r: verify(str(r['answer']), str(r['extracted_prediction'])),
        axis=1,
    )
    merged['is_missing'] = merged['extracted_prediction'].eq('NOT_FOUND')
    merged['raw_output_chars'] = merged['raw_output'].str.len()
    merged['prompt_chars'] = merged['prompt'].astype(str).str.len() if 'prompt' in merged.columns else np.nan
    return merged


merge_df = build_merge_df('./test.csv', 'debug_predictions.csv')
local_acc = merge_df['is_correct'].mean()
print(f'Recomputed accuracy from debug file: {local_acc:.6f}')
print('Rows:', len(merge_df), 'Unique ids:', merge_df['id'].nunique())
display(merge_df.head())

merge_df.to_json(ANALYSIS_DIR / 'merge_df.jsonl', orient='records', lines=True, force_ascii=False)
merge_df.to_csv(ANALYSIS_DIR / 'merge_df.csv', index=False)


## 8. Summary metrics and exported case files


In [ ]:
def make_label_report(df: pd.DataFrame) -> pd.DataFrame:
    """Create per-label count, accuracy, error, and missing-answer statistics."""
    if 'label' not in df.columns:
        out = pd.DataFrame({
            'label': ['ALL'],
            'n': [len(df)],
            'correct': [int(df['is_correct'].sum())],
            'wrong': [int((~df['is_correct']).sum())],
            'accuracy': [df['is_correct'].mean()],
            'error_rate': [1 - df['is_correct'].mean()],
            'missing': [int(df['is_missing'].sum())],
            'missing_rate': [df['is_missing'].mean()],
        })
        return out

    out = (
        df.groupby('label')
        .agg(
            n=('id', 'count'),
            correct=('is_correct', 'sum'),
            wrong=('is_correct', lambda x: int((~x).sum())),
            accuracy=('is_correct', 'mean'),
            missing=('is_missing', 'sum'),
            missing_rate=('is_missing', 'mean'),
            median_raw_chars=('raw_output_chars', 'median'),
        )
        .reset_index()
    )
    out['error_rate'] = 1 - out['accuracy']
    out = out.sort_values(['accuracy', 'n'], ascending=[True, False]).reset_index(drop=True)
    return out


label_report = make_label_report(merge_df)
label_report.to_csv(ANALYSIS_DIR / 'per_label_metrics.csv', index=False)

overall_summary = pd.DataFrame([{
    'n': len(merge_df),
    'correct': int(merge_df['is_correct'].sum()),
    'wrong': int((~merge_df['is_correct']).sum()),
    'accuracy': merge_df['is_correct'].mean(),
    'missing': int(merge_df['is_missing'].sum()),
    'missing_rate': merge_df['is_missing'].mean(),
}])
overall_summary.to_csv(ANALYSIS_DIR / 'overall_summary.csv', index=False)

wrong_df = merge_df[~merge_df['is_correct']].copy()
correct_df = merge_df[merge_df['is_correct']].copy()
missing_df = merge_df[merge_df['is_missing']].copy()

wrong_df.to_csv(ANALYSIS_DIR / 'wrong_cases.csv', index=False)
correct_df.to_csv(ANALYSIS_DIR / 'correct_cases.csv', index=False)
missing_df.to_csv(ANALYSIS_DIR / 'missing_extraction_cases.csv', index=False)

# Export hard categories separately for quick manual review.
if 'label' in merge_df.columns:
    for hard_label in ['cryptarithm_deduce', 'cryptarithm_guess', 'equation_numeric_guess', 'bit_manipulation']:
        subset = wrong_df[wrong_df['label'].eq(hard_label)]
        if len(subset):
            subset.to_csv(ANALYSIS_DIR / f'wrong_{hard_label}.csv', index=False)

print('Overall summary')
display(overall_summary)
print('Per-label report')
display(label_report)
print(f'Wrong cases saved to: {ANALYSIS_DIR / "wrong_cases.csv"}')


## 9. Huikang-style advanced metrics dashboard: 300 DPI figures + matching tables

In [ ]:
import html
from IPython.display import HTML

# =========================
# Huikang-style dashboard configuration
# =========================
PLOT_DPI = int(CFG.get('PLOT_DPI', 300))
TABLE_DIR = ANALYSIS_DIR / CFG.get('TABLE_DIR', 'plot_tables')
TABLE_DIR.mkdir(exist_ok=True, parents=True)
FIGURE_DIR.mkdir(exist_ok=True, parents=True)

ADVANCED_DIR = ANALYSIS_DIR / 'advanced_diagnostics'
ADVANCED_DIR.mkdir(exist_ok=True, parents=True)
WRONG_BY_LABEL_DIR = ANALYSIS_DIR / 'wrong_cases_by_label'
WRONG_BY_LABEL_DIR.mkdir(exist_ok=True, parents=True)

plt.rcParams['savefig.dpi'] = PLOT_DPI

PLOT_REGISTRY = []
TABLE_REGISTRY = []
SKIP_LOG = []


def _safe_display_df(df: pd.DataFrame, max_rows: int = 30) -> pd.DataFrame:
    """Return a display-friendly copy without mutating the exported table."""
    if df is None:
        return pd.DataFrame()
    if len(df) <= max_rows:
        return df
    head = df.head(max_rows).copy()
    note = pd.DataFrame([{col: '...' for col in df.columns}])
    return pd.concat([head, note], ignore_index=True)


def export_table(name: str, df: pd.DataFrame, max_display_rows: int = 30) -> dict:
    """Export a table as CSV and HTML, then display it in the notebook."""
    clean = df.copy() if df is not None else pd.DataFrame()
    csv_path = TABLE_DIR / f'{name}.csv'
    html_path = TABLE_DIR / f'{name}.html'
    csv_path.parent.mkdir(exist_ok=True, parents=True)
    clean.to_csv(csv_path, index=False)
    clean.to_html(html_path, index=False, escape=False)
    print(f'Saved table: {csv_path}')
    display(Markdown(f'**Source table: `{csv_path}`**'))
    display(_safe_display_df(clean, max_display_rows))
    return {'csv': csv_path, 'html': html_path, 'df': clean}


def savefig(path: str | Path):
    """Save a clean figure in both PNG and PDF formats at configured 300 DPI."""
    path = Path(path)
    path.parent.mkdir(exist_ok=True, parents=True)
    plt.tight_layout()
    plt.savefig(path, dpi=PLOT_DPI, bbox_inches='tight')
    plt.savefig(path.with_suffix('.pdf'), dpi=PLOT_DPI, bbox_inches='tight')
    print(f'Saved figure: {path}  |  dpi={PLOT_DPI}')
    return path


def register_plot(title: str, figure_path: Path, table_info: dict, description: str = ''):
    """Register figure/table pairs for the final HTML dashboard."""
    PLOT_REGISTRY.append({
        'title': title,
        'figure_path': Path(figure_path),
        'table_csv': Path(table_info['csv']),
        'table_html': Path(table_info['html']),
        'table_df': table_info['df'],
        'description': description,
    })


def register_table(title: str, table_info: dict, description: str = '', link_path: Path | None = None):
    """Register table-only artifacts for the final HTML dashboard."""
    TABLE_REGISTRY.append({
        'title': title,
        'table_csv': Path(table_info['csv']),
        'table_html': Path(table_info['html']),
        'table_df': table_info['df'],
        'description': description,
        'link_path': Path(link_path) if link_path is not None else None,
    })


def log_skip(name: str, reason: str):
    """Track optional diagnostics that cannot be generated with the available columns/files."""
    SKIP_LOG.append({'diagnostic': name, 'status': 'skipped', 'reason': reason})
    print(f'[SKIP] {name}: {reason}')


def add_percent_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add percent-format metric columns for exported tables."""
    out = df.copy()
    for col in ['accuracy', 'error_rate', 'missing_rate', 'extraction_success_rate', 'wrong_rate', 'disagreement_rate']:
        if col in out.columns:
            out[f'{col}_pct'] = (pd.to_numeric(out[col], errors='coerce') * 100).round(2)
    return out


def label_or_all(df: pd.DataFrame) -> pd.Series:
    """Return the task label column if present; otherwise return a single ALL group."""
    if 'label' in df.columns:
        return df['label'].fillna('UNKNOWN').astype(str)
    return pd.Series(['ALL'] * len(df), index=df.index, name='label')


def _norm_text(x) -> str:
    if pd.isna(x):
        return ''
    return str(x).strip()


def _is_missing_value(x) -> bool:
    s = _norm_text(x)
    return s == '' or s.upper() in {'NOT_FOUND', 'NONE', 'NAN', '<NA>', 'NULL'}


def _looks_int(s: str) -> bool:
    return re.fullmatch(r'[-+]?\d+', s.strip()) is not None


def _looks_float(s: str) -> bool:
    return re.fullmatch(r'[-+]?(?:\d+\.\d*|\.\d+)(?:[eE][-+]?\d+)?', s.strip()) is not None or re.fullmatch(r'[-+]?\d+[eE][-+]?\d+', s.strip()) is not None


def _looks_fraction(s: str) -> bool:
    return re.fullmatch(r'[-+]?\d+\s*/\s*[-+]?\d+', s.strip()) is not None


def answer_format(x) -> str:
    """Coarse answer-format classifier for parser/debug diagnostics."""
    s = _norm_text(x)
    if _is_missing_value(s):
        return 'empty'
    if _looks_int(s):
        return 'integer'
    if _looks_float(s):
        return 'float'
    if _looks_fraction(s):
        return 'fraction'
    if re.fullmatch(r'(true|false|yes|no)', s.lower()):
        return 'boolean'
    if re.fullmatch(r'[A-Za-z]+', s):
        return 'alpha_string'
    if re.fullmatch(r'[A-Za-z0-9_\-]+', s):
        return 'alnum_string'
    if s.startswith('[') or s.startswith('(') or ',' in s:
        return 'list_or_tuple'
    return 'text'


def parse_numeric_value(x):
    """Return a float if the value looks numeric; otherwise NaN."""
    s = _norm_text(x)
    try:
        if _looks_fraction(s):
            a, b = re.split(r'/', s)
            return float(a) / float(b)
        if _looks_int(s) or _looks_float(s):
            return float(s)
    except Exception:
        return np.nan
    return np.nan


def make_run_overview_table() -> pd.DataFrame:
    """Create a compact run-level summary similar to the metrics index page."""
    rows = [
        ('validation_rows', len(merge_df)),
        ('unique_ids', merge_df['id'].nunique() if 'id' in merge_df.columns else len(merge_df)),
        ('correct', int(merge_df['is_correct'].sum())),
        ('wrong', int((~merge_df['is_correct']).sum())),
        ('accuracy', f'{merge_df["is_correct"].mean():.6f}'),
        ('accuracy_pct', f'{merge_df["is_correct"].mean() * 100:.2f}'),
        ('missing_extractions', int(merge_df['is_missing'].sum()) if 'is_missing' in merge_df.columns else 'N/A'),
        ('missing_rate_pct', f'{merge_df["is_missing"].mean() * 100:.2f}' if 'is_missing' in merge_df.columns else 'N/A'),
        ('label_count', merge_df['label'].nunique() if 'label' in merge_df.columns else 'N/A'),
        ('val_path', str(globals().get('VAL_PATH', 'N/A'))),
        ('adapter_dir', str(globals().get('ADAPTER_DIR', 'N/A'))),
        ('submission_zip', str(globals().get('SUBMISSION_ZIP', 'N/A'))),
        ('max_lora_rank', CFG.get('MAX_LORA_RANK')),
        ('max_tokens', CFG.get('MAX_TOKENS')),
        ('temperature', CFG.get('TEMPERATURE')),
        ('top_p', CFG.get('TOP_P')),
        ('max_num_seqs', CFG.get('MAX_NUM_SEQS')),
        ('gpu_memory_utilization', CFG.get('GPU_MEMORY_UTILIZATION')),
        ('max_model_len', CFG.get('MAX_MODEL_LEN')),
        ('figure_dpi', PLOT_DPI),
    ]

    # Add optional runtime fields if a previous cell captured them.
    for key in ['inference_runtime_sec', 'examples_per_second', 'avg_time_per_example_sec']:
        if key in globals():
            rows.append((key, globals()[key]))
    return pd.DataFrame(rows, columns=['field', 'value'])


def plot_run_overview():
    """Export the run overview table; no figure is needed for this block."""
    table = make_run_overview_table()
    info = export_table('00_run_overview', table, max_display_rows=100)
    register_table('Run overview', info, 'Run-level configuration and aggregate local-CV metrics.')
    return info


def make_detailed_task_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Build a richer per-task metrics table used by several plots."""
    work = df.copy()
    work['_label'] = label_or_all(work)
    work['_raw_output_chars'] = pd.to_numeric(work.get('raw_output_chars', np.nan), errors='coerce')
    work['_prompt_chars'] = pd.to_numeric(work.get('prompt_chars', np.nan), errors='coerce')
    work['_is_missing'] = work['is_missing'].astype(bool) if 'is_missing' in work.columns else False

    out = (
        work.groupby('_label', dropna=False)
        .agg(
            n=('id', 'count') if 'id' in work.columns else ('is_correct', 'count'),
            correct=('is_correct', 'sum'),
            wrong=('is_correct', lambda x: int((~x.astype(bool)).sum())),
            accuracy=('is_correct', 'mean'),
            missing=('_is_missing', 'sum'),
            missing_rate=('_is_missing', 'mean'),
            avg_raw_chars=('_raw_output_chars', 'mean'),
            median_raw_chars=('_raw_output_chars', 'median'),
            p90_raw_chars=('_raw_output_chars', lambda x: np.nanpercentile(x.dropna(), 90) if x.dropna().size else np.nan),
            avg_prompt_chars=('_prompt_chars', 'mean'),
            median_prompt_chars=('_prompt_chars', 'median'),
        )
        .reset_index()
        .rename(columns={'_label': 'label'})
    )
    out['error_rate'] = 1 - out['accuracy']
    out['extraction_success_rate'] = 1 - out['missing_rate']
    num_cols = ['accuracy', 'error_rate', 'missing_rate', 'extraction_success_rate', 'avg_raw_chars', 'median_raw_chars', 'p90_raw_chars', 'avg_prompt_chars', 'median_prompt_chars']
    for col in num_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce').round(6)
    out = out.sort_values(['accuracy', 'n'], ascending=[True, False]).reset_index(drop=True)
    return out


def plot_accuracy_by_label(report: pd.DataFrame):
    if len(report) == 0:
        return

    data = add_percent_columns(report).sort_values('accuracy', ascending=True).reset_index(drop=True)
    export_cols = [c for c in [
        'label', 'n', 'correct', 'wrong', 'accuracy', 'accuracy_pct',
        'error_rate', 'error_rate_pct', 'missing', 'missing_rate', 'missing_rate_pct',
        'median_raw_chars'
    ] if c in data.columns]
    table_info = export_table('01_accuracy_by_label', data[export_cols], max_display_rows=100)

    fig_h = max(4.5, 0.45 * len(data) + 1.5)
    plt.figure(figsize=(10, fig_h))
    plt.barh(data['label'], data['accuracy_pct'])
    plt.xlabel('Accuracy (%)')
    plt.ylabel('Task label')
    plt.title('Local CV accuracy by task type')
    plt.xlim(0, 105)
    for i, (_, row) in enumerate(data.iterrows()):
        plt.text(float(row['accuracy_pct']) + 1, i, f"{float(row['accuracy_pct']):.1f}%  n={int(row['n'])}", va='center')
    fig_path = savefig(FIGURE_DIR / '01_accuracy_by_label.png')
    plt.show()

    register_plot(
        title='Accuracy by task type',
        figure_path=fig_path,
        table_info=table_info,
        description='Per-category accuracy, sorted from weakest to strongest.'
    )


def plot_error_count_by_label(report: pd.DataFrame):
    if len(report) == 0 or 'wrong' not in report.columns:
        return

    data = add_percent_columns(report).sort_values('wrong', ascending=True).reset_index(drop=True)
    export_cols = [c for c in [
        'label', 'n', 'correct', 'wrong', 'accuracy_pct', 'error_rate_pct',
        'missing', 'missing_rate_pct'
    ] if c in data.columns]
    table_info = export_table('02_error_count_by_label', data[export_cols], max_display_rows=100)

    fig_h = max(4.5, 0.45 * len(data) + 1.5)
    plt.figure(figsize=(10, fig_h))
    plt.barh(data['label'], data['wrong'])
    plt.xlabel('Wrong cases')
    plt.ylabel('Task label')
    plt.title('Error count by task type')
    xmax = max(1, int(data['wrong'].max()))
    plt.xlim(0, xmax * 1.15 + 1)
    for i, (_, row) in enumerate(data.iterrows()):
        plt.text(float(row['wrong']) + 0.2, i, f"{int(row['wrong'])}/{int(row['n'])}", va='center')
    fig_path = savefig(FIGURE_DIR / '02_error_count_by_label.png')
    plt.show()

    register_plot(
        title='Error count by task type',
        figure_path=fig_path,
        table_info=table_info,
        description='Absolute number of wrong cases per task type.'
    )


def plot_correct_wrong_stacked(report: pd.DataFrame):
    if len(report) == 0 or not {'correct', 'wrong'}.issubset(report.columns):
        return

    data = add_percent_columns(report).sort_values('n', ascending=False).reset_index(drop=True)
    export_cols = [c for c in [
        'label', 'n', 'correct', 'wrong', 'accuracy_pct', 'error_rate_pct'
    ] if c in data.columns]
    table_info = export_table('03_correct_wrong_by_label', data[export_cols], max_display_rows=100)

    x = np.arange(len(data))
    plt.figure(figsize=(max(10, 0.8 * len(data)), 6))
    plt.bar(x, data['correct'], label='Correct')
    plt.bar(x, data['wrong'], bottom=data['correct'], label='Wrong')
    plt.xticks(x, data['label'], rotation=45, ha='right')
    plt.ylabel('Case count')
    plt.title('Correct vs wrong cases by task type')
    plt.legend()
    fig_path = savefig(FIGURE_DIR / '03_correct_wrong_by_label.png')
    plt.show()

    register_plot(
        title='Correct vs wrong cases',
        figure_path=fig_path,
        table_info=table_info,
        description='Stacked count view for each category.'
    )


def plot_missing_rate_by_label(report: pd.DataFrame):
    if len(report) == 0 or 'missing_rate' not in report.columns:
        return

    data = add_percent_columns(report).sort_values('missing_rate', ascending=True).reset_index(drop=True)
    export_cols = [c for c in [
        'label', 'n', 'missing', 'missing_rate', 'missing_rate_pct', 'wrong', 'accuracy_pct'
    ] if c in data.columns]
    table_info = export_table('04_missing_rate_by_label', data[export_cols], max_display_rows=100)

    fig_h = max(4.5, 0.45 * len(data) + 1.5)
    plt.figure(figsize=(10, fig_h))
    plt.barh(data['label'], data['missing_rate_pct'])
    plt.xlabel('Missing extraction rate (%)')
    plt.ylabel('Task label')
    plt.title('Final-answer extraction failure rate')
    xmax = max(1.0, float(data['missing_rate_pct'].max()))
    plt.xlim(0, xmax * 1.20 + 1)
    for i, (_, row) in enumerate(data.iterrows()):
        plt.text(float(row['missing_rate_pct']) + 0.2, i, f"{float(row['missing_rate_pct']):.1f}%", va='center')
    fig_path = savefig(FIGURE_DIR / '04_missing_rate_by_label.png')
    plt.show()

    register_plot(
        title='Final-answer extraction failure rate',
        figure_path=fig_path,
        table_info=table_info,
        description='Cases where no final answer could be extracted.'
    )


def plot_accuracy_vs_count(report: pd.DataFrame):
    if len(report) == 0:
        return

    data = add_percent_columns(report).sort_values('label').reset_index(drop=True)
    export_cols = [c for c in [
        'label', 'n', 'accuracy', 'accuracy_pct', 'correct', 'wrong',
        'missing', 'missing_rate_pct'
    ] if c in data.columns]
    table_info = export_table('05_accuracy_vs_count', data[export_cols], max_display_rows=100)

    plt.figure(figsize=(8, 6))
    sizes = np.maximum(40, data['n'].astype(float) * 1.5)
    plt.scatter(data['n'], data['accuracy_pct'], s=sizes)
    for _, row in data.iterrows():
        plt.text(row['n'], row['accuracy_pct'], row['label'], fontsize=8, ha='left', va='bottom')
    plt.xlabel('Number of cases')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy vs validation-set size by task type')
    plt.ylim(-5, 105)
    fig_path = savefig(FIGURE_DIR / '05_accuracy_vs_count.png')
    plt.show()

    register_plot(
        title='Accuracy vs validation-set size',
        figure_path=fig_path,
        table_info=table_info,
        description='Bubble size is proportional to validation count.'
    )


def plot_output_length_distribution(df: pd.DataFrame):
    if 'raw_output_chars' not in df.columns or len(df) == 0:
        log_skip('raw_output_length_distribution', 'raw_output_chars column is missing.')
        return

    length_series = pd.to_numeric(df['raw_output_chars'], errors='coerce').dropna()
    if length_series.empty:
        log_skip('raw_output_length_distribution', 'raw_output_chars has no numeric values.')
        return

    vmin, vmax = float(length_series.min()), float(length_series.max())
    if vmin == vmax:
        bins = np.array([vmin - 0.5, vmax + 0.5])
    else:
        bins = np.linspace(vmin, vmax, 41)

    correct_len = pd.to_numeric(df.loc[df['is_correct'], 'raw_output_chars'], errors='coerce').dropna()
    wrong_len = pd.to_numeric(df.loc[~df['is_correct'], 'raw_output_chars'], errors='coerce').dropna()

    correct_counts, bin_edges = np.histogram(correct_len, bins=bins)
    wrong_counts, _ = np.histogram(wrong_len, bins=bins)
    hist_table = pd.DataFrame({
        'bin_left_chars': bin_edges[:-1].round(2),
        'bin_right_chars': bin_edges[1:].round(2),
        'correct_count': correct_counts.astype(int),
        'wrong_count': wrong_counts.astype(int),
        'total_count': (correct_counts + wrong_counts).astype(int),
    })
    table_info = export_table('06_raw_output_length_distribution', hist_table, max_display_rows=100)

    plt.figure(figsize=(8, 5))
    plt.hist([correct_len, wrong_len], bins=bins, label=['Correct', 'Wrong'])
    plt.xlabel('Raw output length, characters')
    plt.ylabel('Case count')
    plt.title('Raw output length distribution')
    plt.legend()
    fig_path = savefig(FIGURE_DIR / '06_raw_output_length_distribution.png')
    plt.show()

    register_plot(
        title='Raw output length distribution',
        figure_path=fig_path,
        table_info=table_info,
        description='Histogram source table contains the exact bin edges and counts.'
    )


def export_case_review_tables(df: pd.DataFrame, report: pd.DataFrame):
    """Export review-oriented tables that match the visual metrics."""
    if len(df) == 0:
        return

    wrong_cases = df.loc[~df['is_correct']].copy()
    if len(wrong_cases):
        cols = [c for c in ['id', 'label', 'answer', 'extracted_prediction', 'is_missing', 'raw_output_chars', 'prompt', 'raw_output'] if c in wrong_cases.columns]
        worst_labels = report.sort_values(['accuracy', 'n'], ascending=[True, False])['label'].head(5).tolist() if 'label' in report.columns else []
        review = wrong_cases[wrong_cases['label'].isin(worst_labels)] if 'label' in wrong_cases.columns and worst_labels else wrong_cases
        sort_cols = [c for c in ['label', 'raw_output_chars'] if c in review.columns]
        if sort_cols:
            review = review.sort_values(sort_cols)
        review = review.head(100)
        info = export_table('07_review_wrong_cases_from_weakest_labels', review[cols], max_display_rows=30)
        register_table(
            'Wrong cases from weakest labels',
            info,
            'Top wrong examples from the weakest task types for manual debugging.'
        )

    if 'label' in df.columns:
        by_label_missing = (
            df.loc[df.get('is_missing', False)]
            .groupby('label')
            .agg(missing_cases=('id', 'count'))
            .reset_index()
            .sort_values('missing_cases', ascending=False)
        )
        info = export_table('08_missing_case_count_table', by_label_missing, max_display_rows=100)
        register_table('Missing case count table', info, 'Number of missing extracted answers by label.')


def plot_task_metric_heatmap(task_metrics: pd.DataFrame):
    if task_metrics.empty:
        log_skip('task_metric_heatmap', 'No task metrics available.')
        return

    data = add_percent_columns(task_metrics)
    export_cols = [c for c in [
        'label', 'n', 'correct', 'wrong', 'accuracy_pct', 'error_rate_pct', 'missing_rate_pct',
        'extraction_success_rate_pct', 'avg_raw_chars', 'median_raw_chars', 'p90_raw_chars', 'avg_prompt_chars'
    ] if c in data.columns]
    table_info = export_table('09_task_metric_heatmap', data[export_cols], max_display_rows=100)

    heat_cols = [c for c in ['accuracy_pct', 'error_rate_pct', 'missing_rate_pct', 'extraction_success_rate_pct'] if c in data.columns]
    if not heat_cols:
        log_skip('task_metric_heatmap', 'No numeric percent metrics available for heatmap.')
        return

    matrix = data[heat_cols].astype(float).to_numpy()
    plt.figure(figsize=(max(8, 1.6 * len(heat_cols)), max(4.5, 0.45 * len(data) + 1.5)))
    plt.imshow(matrix, aspect='auto')
    plt.yticks(np.arange(len(data)), data['label'])
    plt.xticks(np.arange(len(heat_cols)), [c.replace('_pct', '').replace('_', ' ') for c in heat_cols], rotation=30, ha='right')
    plt.colorbar(label='Percent')
    plt.title('Per-task metric heatmap')
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            if not np.isnan(val):
                plt.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8)
    fig_path = savefig(FIGURE_DIR / '09_task_metric_heatmap.png')
    plt.show()

    register_plot(
        title='Per-task metric heatmap',
        figure_path=fig_path,
        table_info=table_info,
        description='Compact heatmap of accuracy, error, missing-extraction, and extraction-success rates.'
    )


def classify_error_row(row: pd.Series) -> str:
    """Assign a coarse error class for non-correct cases."""
    if bool(row.get('is_correct', False)):
        return 'correct'

    pred = row.get('extracted_prediction', '')
    gold = row.get('answer', '')
    raw = _norm_text(row.get('raw_output', ''))

    if bool(row.get('is_missing', False)) or _is_missing_value(pred):
        return 'missing_prediction'

    gold_s = _norm_text(gold)
    pred_s = _norm_text(pred)
    gold_fmt = answer_format(gold_s)
    pred_fmt = answer_format(pred_s)

    # If the gold answer appears in the raw output but the extracted prediction is wrong,
    # the most likely issue is answer extraction or final-answer formatting.
    if len(gold_s) >= 2 and gold_s in raw and gold_s != pred_s:
        return 'parser_missed_gold_in_raw'

    gold_num = parse_numeric_value(gold_s)
    pred_num = parse_numeric_value(pred_s)
    if not np.isnan(gold_num) and not np.isnan(pred_num):
        return 'numeric_mismatch'

    if gold_fmt != pred_fmt:
        return 'format_mismatch'

    return 'semantic_wrong'


def add_error_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['gold_format'] = out['answer'].apply(answer_format) if 'answer' in out.columns else 'unknown'
    out['pred_format'] = out['extracted_prediction'].apply(answer_format) if 'extracted_prediction' in out.columns else 'unknown'
    out['error_type'] = out.apply(classify_error_row, axis=1)
    out['gold_numeric'] = out['answer'].apply(parse_numeric_value) if 'answer' in out.columns else np.nan
    out['pred_numeric'] = out['extracted_prediction'].apply(parse_numeric_value) if 'extracted_prediction' in out.columns else np.nan
    out['numeric_abs_error'] = (out['gold_numeric'] - out['pred_numeric']).abs()
    return out


def plot_error_type_distribution(df: pd.DataFrame):
    if 'error_type' not in df.columns:
        log_skip('error_type_distribution', 'error_type column is missing.')
        return

    data = (
        df.loc[~df['is_correct']]
        .groupby('error_type')
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=True)
    )
    if data.empty:
        log_skip('error_type_distribution', 'No wrong cases found.')
        return
    data['percentage'] = (data['count'] / data['count'].sum() * 100).round(2)
    table_info = export_table('10_error_type_distribution', data, max_display_rows=100)

    plt.figure(figsize=(9, max(4, 0.5 * len(data) + 1.5)))
    plt.barh(data['error_type'], data['count'])
    plt.xlabel('Wrong case count')
    plt.ylabel('Error type')
    plt.title('Error taxonomy among wrong cases')
    xmax = max(1, int(data['count'].max()))
    plt.xlim(0, xmax * 1.18 + 1)
    for i, (_, row) in enumerate(data.iterrows()):
        plt.text(row['count'] + 0.2, i, f"{int(row['count'])} ({row['percentage']:.1f}%)", va='center')
    fig_path = savefig(FIGURE_DIR / '10_error_type_distribution.png')
    plt.show()

    register_plot(
        title='Error type distribution',
        figure_path=fig_path,
        table_info=table_info,
        description='Coarse taxonomy of wrong cases: missing prediction, parser issue, numeric mismatch, format mismatch, or semantic wrong.'
    )


def plot_error_type_by_label(df: pd.DataFrame):
    if 'error_type' not in df.columns:
        log_skip('error_type_by_label', 'error_type column is missing.')
        return
    work = df.loc[~df['is_correct']].copy()
    if work.empty:
        log_skip('error_type_by_label', 'No wrong cases found.')
        return
    work['_label'] = label_or_all(work)
    pivot = pd.crosstab(work['_label'], work['error_type'])
    table = pivot.reset_index().rename(columns={'_label': 'label'})
    table_info = export_table('11_error_type_by_label', table, max_display_rows=100)

    matrix = pivot.to_numpy(dtype=float)
    plt.figure(figsize=(max(8, 1.4 * pivot.shape[1]), max(4.5, 0.45 * pivot.shape[0] + 1.5)))
    plt.imshow(matrix, aspect='auto')
    plt.yticks(np.arange(pivot.shape[0]), pivot.index)
    plt.xticks(np.arange(pivot.shape[1]), pivot.columns, rotation=35, ha='right')
    plt.colorbar(label='Wrong case count')
    plt.title('Error type by task label')
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] > 0:
                plt.text(j, i, str(int(matrix[i, j])), ha='center', va='center', fontsize=8)
    fig_path = savefig(FIGURE_DIR / '11_error_type_by_label.png')
    plt.show()

    register_plot(
        title='Error type by task label',
        figure_path=fig_path,
        table_info=table_info,
        description='Cross-tab of task label and error type for wrong cases.'
    )


def plot_extraction_success_by_label(task_metrics: pd.DataFrame):
    if task_metrics.empty or 'extraction_success_rate' not in task_metrics.columns:
        log_skip('extraction_success_by_label', 'extraction_success_rate is unavailable.')
        return
    data = add_percent_columns(task_metrics).sort_values('extraction_success_rate', ascending=True)
    export_cols = [c for c in ['label', 'n', 'missing', 'missing_rate_pct', 'extraction_success_rate_pct', 'wrong', 'accuracy_pct'] if c in data.columns]
    table_info = export_table('12_extraction_success_by_label', data[export_cols], max_display_rows=100)

    plt.figure(figsize=(10, max(4.5, 0.45 * len(data) + 1.5)))
    plt.barh(data['label'], data['extraction_success_rate_pct'])
    plt.xlabel('Extraction success rate (%)')
    plt.ylabel('Task label')
    plt.title('Answer extraction success by task type')
    plt.xlim(0, 105)
    for i, (_, row) in enumerate(data.iterrows()):
        plt.text(float(row['extraction_success_rate_pct']) + 1, i, f"{float(row['extraction_success_rate_pct']):.1f}%", va='center')
    fig_path = savefig(FIGURE_DIR / '12_extraction_success_by_label.png')
    plt.show()

    register_plot(
        title='Answer extraction success by task type',
        figure_path=fig_path,
        table_info=table_info,
        description='Checks whether failures are caused by missing final-answer extraction rather than reasoning ability.'
    )


def plot_output_length_vs_correctness(df: pd.DataFrame):
    if 'raw_output_chars' not in df.columns:
        log_skip('output_length_vs_correctness', 'raw_output_chars column is missing.')
        return
    work = df.copy()
    work['raw_output_chars'] = pd.to_numeric(work['raw_output_chars'], errors='coerce')
    work['correctness'] = np.where(work['is_correct'], 'correct', np.where(work.get('is_missing', False), 'missing_prediction', 'wrong'))
    stats = (
        work.groupby('correctness')
        .agg(
            n=('raw_output_chars', 'count'),
            mean_len=('raw_output_chars', 'mean'),
            median_len=('raw_output_chars', 'median'),
            p90_len=('raw_output_chars', lambda x: np.nanpercentile(x.dropna(), 90) if x.dropna().size else np.nan),
            max_len=('raw_output_chars', 'max'),
        )
        .reset_index()
    )
    for c in ['mean_len', 'median_len', 'p90_len', 'max_len']:
        stats[c] = pd.to_numeric(stats[c], errors='coerce').round(2)
    table_info = export_table('13_output_length_vs_correctness', stats, max_display_rows=100)

    groups = [g for g in ['correct', 'wrong', 'missing_prediction'] if g in set(work['correctness'])]
    values = [work.loc[work['correctness'].eq(g), 'raw_output_chars'].dropna() for g in groups]
    if not values or all(v.empty for v in values):
        log_skip('output_length_vs_correctness', 'No non-empty output length groups.')
        return
    plt.figure(figsize=(8, 5))
    plt.boxplot(values, labels=groups, showfliers=False)
    plt.ylabel('Raw output length, characters')
    plt.title('Output length vs correctness')
    fig_path = savefig(FIGURE_DIR / '13_output_length_vs_correctness.png')
    plt.show()

    register_plot(
        title='Output length vs correctness',
        figure_path=fig_path,
        table_info=table_info,
        description='Length diagnostics split by correct, wrong, and missing-prediction cases.'
    )


def add_difficulty_proxies(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    text_col = 'prompt' if 'prompt' in out.columns else ('question' if 'question' in out.columns else None)
    if text_col is None:
        out['question_chars'] = np.nan
        out['digit_count'] = np.nan
        out['operator_count'] = np.nan
        return out
    text = out[text_col].fillna('').astype(str)
    out['question_chars'] = text.str.len()
    out['digit_count'] = text.str.count(r'\d')
    out['operator_count'] = text.str.count(r'[+\-*/=<>^%]')
    out['text_col_used_for_difficulty'] = text_col
    return out


def plot_accuracy_by_difficulty_proxy(df: pd.DataFrame):
    work = add_difficulty_proxies(df)
    proxy_cols = [c for c in ['question_chars', 'digit_count', 'operator_count'] if c in work.columns and pd.to_numeric(work[c], errors='coerce').notna().sum() > 0]
    if not proxy_cols:
        log_skip('accuracy_by_difficulty_proxy', 'No prompt/question text available to build difficulty proxies.')
        return

    all_tables = []
    for col in proxy_cols:
        vals = pd.to_numeric(work[col], errors='coerce')
        unique_count = vals.dropna().nunique()
        if unique_count < 2:
            continue
        try:
            bins = pd.qcut(vals, q=min(4, unique_count), duplicates='drop')
        except ValueError:
            bins = pd.cut(vals, bins=min(4, unique_count), duplicates='drop')
        temp = work.copy()
        temp['_bin'] = bins.astype(str)
        tab = (
            temp.dropna(subset=['_bin'])
            .groupby('_bin')
            .agg(
                n=('is_correct', 'count'),
                correct=('is_correct', 'sum'),
                accuracy=('is_correct', 'mean'),
                min_value=(col, 'min'),
                max_value=(col, 'max'),
                mean_value=(col, 'mean'),
            )
            .reset_index()
            .rename(columns={'_bin': 'difficulty_bin'})
        )
        tab.insert(0, 'proxy', col)
        all_tables.append(tab)

    if not all_tables:
        log_skip('accuracy_by_difficulty_proxy', 'Difficulty proxies have insufficient variation.')
        return

    table = pd.concat(all_tables, ignore_index=True)
    table['accuracy_pct'] = (table['accuracy'] * 100).round(2)
    for c in ['min_value', 'max_value', 'mean_value']:
        table[c] = pd.to_numeric(table[c], errors='coerce').round(2)
    table_info = export_table('14_accuracy_by_difficulty_proxy', table, max_display_rows=100)

    plt.figure(figsize=(max(9, 2.8 * table['proxy'].nunique()), 5))
    x_labels = table['proxy'] + '\n' + table['difficulty_bin']
    x = np.arange(len(table))
    plt.bar(x, table['accuracy_pct'])
    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy by difficulty proxy bins')
    plt.ylim(0, 105)
    fig_path = savefig(FIGURE_DIR / '14_accuracy_by_difficulty_proxy.png')
    plt.show()

    register_plot(
        title='Accuracy by difficulty proxy',
        figure_path=fig_path,
        table_info=table_info,
        description='Accuracy split by prompt length, digit count, and operator count when these proxies can be computed.'
    )


def plot_answer_format_confusion(df: pd.DataFrame):
    if not {'gold_format', 'pred_format'}.issubset(df.columns):
        log_skip('answer_format_confusion', 'gold_format/pred_format columns are missing.')
        return
    pivot = pd.crosstab(df['gold_format'], df['pred_format'])
    if pivot.empty:
        log_skip('answer_format_confusion', 'No answer-format data available.')
        return
    table = pivot.reset_index().rename(columns={'gold_format': 'gold_format'})
    table_info = export_table('15_answer_format_confusion', table, max_display_rows=100)

    matrix = pivot.to_numpy(dtype=float)
    plt.figure(figsize=(max(7, 0.85 * pivot.shape[1] + 3), max(5, 0.55 * pivot.shape[0] + 2)))
    plt.imshow(matrix, aspect='auto')
    plt.yticks(np.arange(pivot.shape[0]), pivot.index)
    plt.xticks(np.arange(pivot.shape[1]), pivot.columns, rotation=35, ha='right')
    plt.colorbar(label='Case count')
    plt.xlabel('Predicted answer format')
    plt.ylabel('Gold answer format')
    plt.title('Gold vs predicted answer-format confusion')
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] > 0:
                plt.text(j, i, str(int(matrix[i, j])), ha='center', va='center', fontsize=8)
    fig_path = savefig(FIGURE_DIR / '15_answer_format_confusion.png')
    plt.show()

    register_plot(
        title='Gold vs predicted answer-format confusion',
        figure_path=fig_path,
        table_info=table_info,
        description='Detects whether failures are caused by wrong answer format instead of wrong reasoning.'
    )


def write_weakest_case_gallery(df: pd.DataFrame, task_metrics: pd.DataFrame, max_labels: int = 5, max_cases_per_label: int = 15):
    wrong = df.loc[~df['is_correct']].copy()
    if wrong.empty:
        log_skip('weakest_case_gallery', 'No wrong cases found.')
        return
    if 'label' in wrong.columns and 'label' in task_metrics.columns:
        labels = task_metrics.sort_values(['accuracy', 'n'], ascending=[True, False])['label'].head(max_labels).tolist()
        wrong = wrong[wrong['label'].isin(labels)]
    else:
        labels = ['ALL']
        wrong['_label'] = 'ALL'

    sort_cols = [c for c in ['label', 'error_type', 'raw_output_chars'] if c in wrong.columns]
    if sort_cols:
        wrong = wrong.sort_values(sort_cols)

    rows = []
    for label_value, sub in wrong.groupby('label' if 'label' in wrong.columns else '_label'):
        rows.append(sub.head(max_cases_per_label))
    gallery_df = pd.concat(rows, ignore_index=True) if rows else wrong.head(max_cases_per_label)
    cols = [c for c in ['id', 'label', 'error_type', 'gold_format', 'pred_format', 'answer', 'extracted_prediction', 'is_missing', 'raw_output_chars', 'prompt', 'raw_output'] if c in gallery_df.columns]
    gallery_df = gallery_df[cols]
    table_info = export_table('16_weakest_label_wrong_case_gallery', gallery_df, max_display_rows=40)

    html_path = ANALYSIS_DIR / 'weakest_label_wrong_cases.html'
    blocks = []
    group_col = 'label' if 'label' in gallery_df.columns else None
    grouped = gallery_df.groupby(group_col) if group_col else [('ALL', gallery_df)]
    for label_value, sub in grouped:
        case_blocks = []
        for _, row in sub.iterrows():
            prompt = html.escape(_norm_text(row.get('prompt', '')))
            raw = html.escape(_norm_text(row.get('raw_output', '')))
            case_blocks.append(f"""
            <article class="case-card">
              <h3>ID: {html.escape(_norm_text(row.get('id', 'N/A')))}</h3>
              <p><b>Error type:</b> {html.escape(_norm_text(row.get('error_type', 'N/A')))}</p>
              <p><b>Gold:</b> <code>{html.escape(_norm_text(row.get('answer', '')))}</code> &nbsp; <b>Pred:</b> <code>{html.escape(_norm_text(row.get('extracted_prediction', '')))}</code></p>
              <details open><summary>Prompt</summary><pre>{prompt}</pre></details>
              <details><summary>Raw model output</summary><pre>{raw}</pre></details>
            </article>
            """)
        blocks.append(f"<section><h2>{html.escape(str(label_value))}</h2>{''.join(case_blocks)}</section>")

    doc = f"""
    <!doctype html>
    <html lang="en">
    <head>
      <meta charset="utf-8">
      <title>Weakest Label Wrong Cases</title>
      <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; margin: 28px; background: #fafafa; color: #222; }}
        .case-card {{ background: white; border: 1px solid #ddd; border-radius: 10px; padding: 14px 16px; margin: 14px 0; }}
        pre {{ white-space: pre-wrap; word-break: break-word; background: #f6f8fa; padding: 10px; border-radius: 6px; max-height: 420px; overflow: auto; }}
        code {{ background: #f3f3f3; padding: 1px 4px; border-radius: 4px; }}
      </style>
    </head>
    <body>
      <h1>Wrong cases from weakest task labels</h1>
      <p>Source table: <code>{html.escape(str(table_info['csv']))}</code></p>
      {''.join(blocks)}
    </body>
    </html>
    """
    html_path.write_text(doc, encoding='utf-8')
    print('Saved weakest-case gallery:', html_path)
    display(Markdown(f'Weakest-case gallery saved to: `{html_path}`'))
    register_table(
        'Weakest-label wrong case gallery',
        table_info,
        'HTML gallery for manual inspection of weak-category failures.',
        link_path=html_path,
    )


def export_wrong_cases_by_label(df: pd.DataFrame):
    wrong = df.loc[~df['is_correct']].copy()
    if wrong.empty:
        log_skip('wrong_cases_by_label', 'No wrong cases found.')
        return
    label_series = label_or_all(wrong)
    wrong['_label_for_export'] = label_series
    export_rows = []
    for label_value, sub in wrong.groupby('_label_for_export'):
        safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(label_value)).strip('_') or 'UNKNOWN'
        out_path = WRONG_BY_LABEL_DIR / f'{safe}_wrong.csv'
        sub.drop(columns=['_label_for_export'], errors='ignore').to_csv(out_path, index=False)
        export_rows.append({'label': label_value, 'wrong_cases': len(sub), 'csv_path': str(out_path)})
    table = pd.DataFrame(export_rows).sort_values('wrong_cases', ascending=False)
    info = export_table('17_wrong_cases_by_label_index', table, max_display_rows=100)
    register_table('Wrong cases by label export index', info, 'One CSV file per label under analysis_outputs/wrong_cases_by_label/.')


def discover_version_result_files() -> list[Path]:
    """Discover optional multi-version result files without requiring a fixed path."""
    configured = CFG.get('VERSION_RESULT_FILES') or CFG.get('VERSION_RESULTS') or []
    paths = []
    if isinstance(configured, (str, Path)):
        configured = [configured]
    for item in configured:
        p = Path(item)
        if p.exists():
            paths.append(p)

    search_dirs = [ANALYSIS_DIR / 'version_results', Path('version_results')]
    for d in search_dirs:
        if d.exists():
            paths.extend(sorted(d.glob('*.csv')))
            paths.extend(sorted(d.glob('*.jsonl')))
    # Deduplicate while preserving order.
    seen = set()
    unique = []
    for p in paths:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def load_version_file(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == '.jsonl':
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_csv(path)
    df['version'] = df.get('version', path.stem)
    return df


def summarize_version_df(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize a version file that already contains is_correct or can be merged with local truth."""
    work = df.copy()
    if 'is_correct' not in work.columns:
        if {'id', 'extracted_prediction'}.issubset(work.columns) and {'id', 'answer'}.issubset(merge_df.columns):
            truth_cols = [c for c in ['id', 'answer', 'label'] if c in merge_df.columns]
            work = work.merge(merge_df[truth_cols].drop_duplicates('id'), on='id', how='left', suffixes=('', '_truth'))
            work['is_correct'] = work.apply(lambda r: verify(str(r.get('answer', '')), str(r.get('extracted_prediction', ''))), axis=1)
        else:
            raise ValueError('Version file needs is_correct, or id + extracted_prediction that can be merged with merge_df.')
    work['_label'] = label_or_all(work)
    version = str(work['version'].iloc[0]) if 'version' in work.columns and len(work) else 'unknown'
    out = (
        work.groupby('_label')
        .agg(n=('is_correct', 'count'), correct=('is_correct', 'sum'), accuracy=('is_correct', 'mean'))
        .reset_index()
        .rename(columns={'_label': 'label'})
    )
    out['version'] = version
    out['wrong'] = out['n'] - out['correct']
    return out[['version', 'label', 'n', 'correct', 'wrong', 'accuracy']]


def plot_version_comparison():
    files = discover_version_result_files()
    if not files:
        log_skip('version_comparison', 'No version result files found. Put CSV/JSONL files under analysis_outputs/version_results/ or set CFG["VERSION_RESULT_FILES"].')
        return
    summaries = []
    for p in files:
        try:
            summaries.append(summarize_version_df(load_version_file(p)))
        except Exception as e:
            log_skip(f'version_file:{p}', str(e))
    if not summaries:
        log_skip('version_comparison', 'No valid version result files could be summarized.')
        return
    table = pd.concat(summaries, ignore_index=True)
    table['accuracy_pct'] = (table['accuracy'] * 100).round(2)
    table_info = export_table('18_version_comparison_by_label', table.sort_values(['label', 'version']), max_display_rows=200)

    pivot = table.pivot_table(index='label', columns='version', values='accuracy_pct', aggfunc='mean')
    plt.figure(figsize=(max(8, 1.1 * pivot.shape[1] + 4), max(4.5, 0.45 * pivot.shape[0] + 1.5)))
    plt.imshow(pivot.to_numpy(dtype=float), aspect='auto')
    plt.yticks(np.arange(pivot.shape[0]), pivot.index)
    plt.xticks(np.arange(pivot.shape[1]), pivot.columns, rotation=35, ha='right')
    plt.colorbar(label='Accuracy (%)')
    plt.title('Version comparison by task label')
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            if not pd.isna(val):
                plt.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8)
    fig_path = savefig(FIGURE_DIR / '18_version_comparison_by_label.png')
    plt.show()
    register_plot('Version comparison by task label', fig_path, table_info, 'Accuracy table and heatmap across multiple local-CV result versions.')

    # Delta heatmap relative to the first version in column order.
    if pivot.shape[1] >= 2:
        base_col = pivot.columns[0]
        delta = pivot.subtract(pivot[base_col], axis=0)
        delta_table = delta.reset_index()
        delta_info = export_table('19_version_delta_heatmap', delta_table, max_display_rows=200)
        plt.figure(figsize=(max(8, 1.1 * delta.shape[1] + 4), max(4.5, 0.45 * delta.shape[0] + 1.5)))
        plt.imshow(delta.to_numpy(dtype=float), aspect='auto')
        plt.yticks(np.arange(delta.shape[0]), delta.index)
        plt.xticks(np.arange(delta.shape[1]), delta.columns, rotation=35, ha='right')
        plt.colorbar(label=f'Accuracy delta vs {base_col} (pp)')
        plt.title('Version accuracy delta heatmap')
        for i in range(delta.shape[0]):
            for j in range(delta.shape[1]):
                val = delta.iloc[i, j]
                if not pd.isna(val):
                    plt.text(j, i, f'{val:+.1f}', ha='center', va='center', fontsize=8)
        fig_path = savefig(FIGURE_DIR / '19_version_delta_heatmap.png')
        plt.show()
        register_plot('Version delta heatmap', fig_path, delta_info, f'Accuracy-point delta relative to {base_col}.')


def discover_training_data_files() -> list[Path]:
    configured = CFG.get('TRAIN_DATA_FILES') or CFG.get('TRAIN_DATA_PATHS') or []
    if isinstance(configured, (str, Path)):
        configured = [configured]
    paths = []
    for item in configured:
        p = Path(item)
        if p.exists():
            paths.append(p)
    search_dirs = [ANALYSIS_DIR / 'training_data_sources', Path('training_data_sources')]
    for d in search_dirs:
        if d.exists():
            paths.extend(sorted(d.glob('*.csv')))
            paths.extend(sorted(d.glob('*.jsonl')))
    seen = set()
    unique = []
    for p in paths:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def load_training_data_for_source_report() -> pd.DataFrame | None:
    files = discover_training_data_files()
    if files:
        dfs = []
        for p in files:
            try:
                if p.suffix.lower() == '.jsonl':
                    tmp = pd.read_json(p, lines=True)
                else:
                    tmp = pd.read_csv(p)
                tmp['_file'] = str(p)
                dfs.append(tmp)
            except Exception as e:
                log_skip(f'training_data_file:{p}', str(e))
        if dfs:
            return pd.concat(dfs, ignore_index=True)

    # Fallback: if the current validation/merge table has source-like fields, use it as a diagnostic table.
    source_like = [c for c in ['source', 'data_source', 'dataset', 'origin', 'split'] if c in merge_df.columns]
    if source_like:
        return merge_df.copy()
    return None


def plot_training_data_source_distribution():
    data = load_training_data_for_source_report()
    if data is None or data.empty:
        log_skip('training_data_source_distribution', 'No training/source data found. Set CFG["TRAIN_DATA_FILES"] or create analysis_outputs/training_data_sources/*.csv/jsonl.')
        return
    source_col = next((c for c in ['source', 'data_source', 'dataset', 'origin', '_file', 'split'] if c in data.columns), None)
    label_col = next((c for c in ['label', 'category', 'task', 'task_type'] if c in data.columns), None)
    if source_col is None:
        log_skip('training_data_source_distribution', 'No source/data_source/dataset/origin column found.')
        return
    table = (
        data.groupby(source_col)
        .size()
        .reset_index(name='n')
        .rename(columns={source_col: 'source'})
        .sort_values('n', ascending=False)
    )
    table['percentage'] = (table['n'] / table['n'].sum() * 100).round(2)
    if label_col is not None:
        covered = data.groupby(source_col)[label_col].nunique().reset_index(name='labels_covered').rename(columns={source_col: 'source'})
        table = table.merge(covered, on='source', how='left')
    table_info = export_table('20_training_data_source_distribution', table, max_display_rows=100)

    plt.figure(figsize=(10, max(4.5, 0.45 * len(table) + 1.5)))
    plot_data = table.sort_values('n', ascending=True)
    plt.barh(plot_data['source'].astype(str), plot_data['n'])
    plt.xlabel('Record count')
    plt.ylabel('Data source')
    plt.title('Training/source data distribution')
    for i, (_, row) in enumerate(plot_data.iterrows()):
        plt.text(row['n'] + max(1, table['n'].max() * 0.01), i, f"{int(row['n'])} ({row['percentage']:.1f}%)", va='center')
    fig_path = savefig(FIGURE_DIR / '20_training_data_source_distribution.png')
    plt.show()
    register_plot('Training/source data distribution', fig_path, table_info, 'Data volume by source; useful for explaining data-mixture changes.')

    if label_col is not None:
        pivot = pd.crosstab(data[source_col].astype(str), data[label_col].astype(str))
        pivot_table = pivot.reset_index().rename(columns={source_col: 'source'})
        pivot_info = export_table('21_training_data_label_source_heatmap', pivot_table, max_display_rows=200)
        plt.figure(figsize=(max(9, 0.7 * pivot.shape[1] + 4), max(4.5, 0.45 * pivot.shape[0] + 1.5)))
        plt.imshow(pivot.to_numpy(dtype=float), aspect='auto')
        plt.yticks(np.arange(pivot.shape[0]), pivot.index)
        plt.xticks(np.arange(pivot.shape[1]), pivot.columns, rotation=35, ha='right')
        plt.colorbar(label='Record count')
        plt.title('Data source × label coverage')
        fig_path = savefig(FIGURE_DIR / '21_training_data_label_source_heatmap.png')
        plt.show()
        register_plot('Data source × label coverage', fig_path, pivot_info, 'Heatmap of training/source rows by source and label.')


def plot_inference_cost_dashboard(df: pd.DataFrame):
    # Works with optional timing/token columns if your debug_predictions.csv includes them.
    possible_time_cols = [c for c in ['latency_sec', 'time_sec', 'runtime_sec', 'elapsed_sec', 'generation_time_sec'] if c in df.columns]
    possible_token_cols = [c for c in ['output_tokens', 'generated_tokens', 'num_output_tokens', 'completion_tokens'] if c in df.columns]
    if not possible_time_cols and not possible_token_cols:
        log_skip('inference_cost_dashboard', 'No per-example latency/token columns found.')
        return

    work = df.copy()
    work['_label'] = label_or_all(work)
    agg_kwargs = {'n': ('is_correct', 'count')}
    time_col = possible_time_cols[0] if possible_time_cols else None
    token_col = possible_token_cols[0] if possible_token_cols else None
    if time_col:
        work[time_col] = pd.to_numeric(work[time_col], errors='coerce')
        agg_kwargs.update(avg_latency_sec=(time_col, 'mean'), median_latency_sec=(time_col, 'median'), total_latency_sec=(time_col, 'sum'))
    if token_col:
        work[token_col] = pd.to_numeric(work[token_col], errors='coerce')
        agg_kwargs.update(avg_output_tokens=(token_col, 'mean'), median_output_tokens=(token_col, 'median'), total_output_tokens=(token_col, 'sum'))
    table = work.groupby('_label').agg(**agg_kwargs).reset_index().rename(columns={'_label': 'label'})
    for c in table.columns:
        if c != 'label':
            table[c] = pd.to_numeric(table[c], errors='coerce').round(4)
    table_info = export_table('22_inference_cost_by_label', table, max_display_rows=100)

    plot_col = 'avg_latency_sec' if 'avg_latency_sec' in table.columns else 'avg_output_tokens'
    plot_data = table.sort_values(plot_col, ascending=True)
    plt.figure(figsize=(10, max(4.5, 0.45 * len(plot_data) + 1.5)))
    plt.barh(plot_data['label'], plot_data[plot_col])
    plt.xlabel(plot_col.replace('_', ' '))
    plt.ylabel('Task label')
    plt.title('Inference cost by task label')
    fig_path = savefig(FIGURE_DIR / '22_inference_cost_by_label.png')
    plt.show()
    register_plot('Inference cost by task label', fig_path, table_info, 'Optional latency/token diagnostic if debug predictions include cost columns.')


def discover_self_consistency_files() -> list[Path]:
    configured = CFG.get('SELF_CONSISTENCY_FILES') or CFG.get('SC_RESULT_FILES') or []
    if isinstance(configured, (str, Path)):
        configured = [configured]
    paths = []
    for item in configured:
        p = Path(item)
        if p.exists():
            paths.append(p)
    search_dirs = [ANALYSIS_DIR / 'self_consistency', Path('self_consistency')]
    for d in search_dirs:
        if d.exists():
            paths.extend(sorted(d.glob('*.csv')))
            paths.extend(sorted(d.glob('*.jsonl')))
    seen = set()
    unique = []
    for p in paths:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def load_self_consistency_table() -> pd.DataFrame | None:
    files = discover_self_consistency_files()
    dfs = []
    for p in files:
        try:
            tmp = pd.read_json(p, lines=True) if p.suffix.lower() == '.jsonl' else pd.read_csv(p)
            tmp['_file'] = str(p)
            dfs.append(tmp)
        except Exception as e:
            log_skip(f'self_consistency_file:{p}', str(e))
    if dfs:
        return pd.concat(dfs, ignore_index=True)

    # Fallback: current merge_df might already include vote_* columns.
    vote_cols = [c for c in merge_df.columns if re.fullmatch(r'(vote|pred|sample)_\d+', str(c))]
    if vote_cols:
        return merge_df.copy()
    return None


def plot_self_consistency_diagnostics():
    sc = load_self_consistency_table()
    if sc is None or sc.empty:
        log_skip('self_consistency_diagnostics', 'No self-consistency files or vote_* columns found.')
        return

    vote_cols = [c for c in sc.columns if re.fullmatch(r'(vote|pred|sample)_\d+', str(c))]
    if not vote_cols:
        log_skip('self_consistency_diagnostics', 'No vote_1/vote_2/... columns found.')
        return

    work = sc.copy()
    if 'label' not in work.columns and {'id', 'label'}.issubset(merge_df.columns):
        work = work.merge(merge_df[['id', 'label']].drop_duplicates('id'), on='id', how='left')
    if 'answer' not in work.columns and {'id', 'answer'}.issubset(merge_df.columns):
        work = work.merge(merge_df[['id', 'answer']].drop_duplicates('id'), on='id', how='left')

    final_col = next((c for c in ['final_vote', 'majority_vote', 'sc_prediction', 'extracted_prediction'] if c in work.columns), None)
    if final_col is None:
        # Majority vote from vote columns.
        work['final_vote'] = work[vote_cols].mode(axis=1, dropna=True)[0]
        final_col = 'final_vote'

    if 'single_prediction' in work.columns:
        single_col = 'single_prediction'
    elif 'baseline_prediction' in work.columns:
        single_col = 'baseline_prediction'
    elif 'extracted_prediction' in merge_df.columns and 'id' in work.columns:
        work = work.merge(merge_df[['id', 'extracted_prediction']].rename(columns={'extracted_prediction': 'single_prediction'}), on='id', how='left')
        single_col = 'single_prediction'
    else:
        single_col = None

    if 'answer' not in work.columns:
        log_skip('self_consistency_diagnostics', 'No answer column available to score votes.')
        return

    work['sc_correct'] = work.apply(lambda r: verify(str(r.get('answer', '')), str(r.get(final_col, ''))), axis=1)
    if single_col:
        work['single_correct'] = work.apply(lambda r: verify(str(r.get('answer', '')), str(r.get(single_col, ''))), axis=1)
    else:
        work['single_correct'] = np.nan

    def disagreement(row):
        vals = [_norm_text(row.get(c, '')) for c in vote_cols]
        vals = [v for v in vals if v != '']
        return 0.0 if len(vals) <= 1 else float(len(set(vals)) > 1)

    work['vote_disagreement'] = work.apply(disagreement, axis=1)
    work['_label'] = label_or_all(work)
    agg = (
        work.groupby('_label')
        .agg(
            n=('sc_correct', 'count'),
            sc_acc=('sc_correct', 'mean'),
            single_acc=('single_correct', 'mean'),
            disagreement_rate=('vote_disagreement', 'mean'),
        )
        .reset_index()
        .rename(columns={'_label': 'label'})
    )
    agg['delta'] = agg['sc_acc'] - agg['single_acc']
    agg['sc_acc_pct'] = (agg['sc_acc'] * 100).round(2)
    agg['single_acc_pct'] = (agg['single_acc'] * 100).round(2)
    agg['delta_pct_point'] = (agg['delta'] * 100).round(2)
    agg['disagreement_rate_pct'] = (agg['disagreement_rate'] * 100).round(2)
    table_info = export_table('23_self_consistency_gain_by_label', agg, max_display_rows=100)

    plot_data = agg.sort_values('delta_pct_point', ascending=True)
    plt.figure(figsize=(10, max(4.5, 0.45 * len(plot_data) + 1.5)))
    plt.barh(plot_data['label'], plot_data['delta_pct_point'])
    plt.xlabel('Self-consistency gain vs single pass, percentage points')
    plt.ylabel('Task label')
    plt.title('Self-consistency gain by task label')
    fig_path = savefig(FIGURE_DIR / '23_self_consistency_gain_by_label.png')
    plt.show()
    register_plot('Self-consistency gain by task label', fig_path, table_info, 'Optional k-sample voting gain if vote columns/files are available.')

    vote_cols_export = [c for c in ['id', 'label', 'answer', single_col, final_col, 'single_correct', 'sc_correct', 'vote_disagreement'] + vote_cols if c and c in work.columns]
    votes_info = export_table('24_self_consistency_vote_table', work[vote_cols_export], max_display_rows=60)
    register_table('Self-consistency vote table', votes_info, 'Per-example vote traces and final majority prediction.')


def make_metrics_dashboard_html(run_table_info: dict):
    """Create a lightweight HTML dashboard with figure/table pairs and table-only artifacts."""
    html_path = ANALYSIS_DIR / 'local_cv_metrics_dashboard.html'

    def rel_from_analysis(path: Path) -> str:
        try:
            return html.escape(os.path.relpath(Path(path), start=ANALYSIS_DIR))
        except Exception:
            return html.escape(str(path))

    run_table_html = run_table_info['df'].to_html(index=False, escape=False, classes='metrics-table')

    plot_blocks = []
    for item in PLOT_REGISTRY:
        table_html = item['table_df'].to_html(index=False, escape=False, classes='metrics-table')
        plot_blocks.append(
            f"""
            <section class="metric-block">
              <h2>{html.escape(item['title'])}</h2>
              <p>{html.escape(item.get('description', ''))}</p>
              <p class="paths">
                Figure: <code>{html.escape(str(item['figure_path']))}</code><br>
                Table: <code>{html.escape(str(item['table_csv']))}</code>
              </p>
              <img src="{rel_from_analysis(item['figure_path'])}" alt="{html.escape(item['title'])}">
              <details open>
                <summary>Source table</summary>
                {table_html}
              </details>
            </section>
            """
        )

    table_blocks = []
    for item in TABLE_REGISTRY:
        table_html = item['table_df'].to_html(index=False, escape=False, classes='metrics-table')
        link_html = ''
        if item.get('link_path') is not None:
            link_html = f'<p class="paths">Artifact: <a href="{rel_from_analysis(item["link_path"])}">{html.escape(str(item["link_path"]))}</a></p>'
        table_blocks.append(
            f"""
            <section class="metric-block">
              <h2>{html.escape(item['title'])}</h2>
              <p>{html.escape(item.get('description', ''))}</p>
              <p class="paths">Table: <code>{html.escape(str(item['table_csv']))}</code></p>
              {link_html}
              <details open>
                <summary>Source table</summary>
                {table_html}
              </details>
            </section>
            """
        )

    skip_html = ''
    if SKIP_LOG:
        skip_table = pd.DataFrame(SKIP_LOG)
        skip_html = f"""
        <section class="summary-block">
          <h2>Skipped optional diagnostics</h2>
          <p>These blocks need optional columns or files. The notebook skipped them instead of failing.</p>
          {skip_table.to_html(index=False, escape=False, classes='metrics-table')}
        </section>
        """

    doc = f"""
    <!doctype html>
    <html lang="en">
    <head>
      <meta charset="utf-8">
      <title>Nemotron Local CV Metrics Dashboard</title>
      <style>
        body {{
          font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
          margin: 32px;
          color: #222;
          background: #fafafa;
        }}
        nav {{ margin-bottom: 20px; font-size: 14px; }}
        nav a {{ margin-right: 12px; color: #2454a6; text-decoration: none; }}
        h1 {{ margin-bottom: 4px; }}
        .subtitle {{ color: #666; margin-top: 0; }}
        .metric-block, .summary-block {{
          background: white;
          border: 1px solid #ddd;
          border-radius: 10px;
          padding: 18px 20px;
          margin: 18px 0;
          box-shadow: 0 1px 2px rgba(0,0,0,0.04);
        }}
        img {{
          max-width: 100%;
          border: 1px solid #e6e6e6;
          border-radius: 6px;
          background: white;
          margin: 10px 0 16px 0;
        }}
        table.metrics-table {{
          border-collapse: collapse;
          width: 100%;
          font-size: 13px;
          margin-top: 8px;
        }}
        table.metrics-table th, table.metrics-table td {{
          border: 1px solid #ddd;
          padding: 6px 8px;
          text-align: left;
          vertical-align: top;
        }}
        table.metrics-table th {{ background: #f1f3f5; position: sticky; top: 0; }}
        details {{ max-height: 560px; overflow: auto; }}
        .paths {{ color: #666; font-size: 13px; }}
        code {{ background: #f3f3f3; padding: 1px 4px; border-radius: 4px; }}
      </style>
    </head>
    <body>
      <nav>
        <a href="#summary">Summary</a>
        <a href="#plots">Figures</a>
        <a href="#tables">Tables</a>
        <a href="plot_tables/00_run_overview.csv">Run table CSV</a>
      </nav>

      <h1>Nemotron Local CV Metrics Dashboard</h1>
      <p class="subtitle">Huikang-style metric blocks: every 300-DPI figure has a matching exported source table.</p>

      <section id="summary" class="summary-block">
        <h2>Run overview</h2>
        {run_table_html}
      </section>

      <section id="plots">
        {''.join(plot_blocks)}
      </section>

      <section id="tables">
        {''.join(table_blocks)}
      </section>

      {skip_html}
    </body>
    </html>
    """

    html_path.write_text(doc, encoding='utf-8')
    print(f'Saved HTML dashboard: {html_path}')
    display(Markdown(f'HTML dashboard saved to: `{html_path}`'))
    return html_path


# =========================
# Generate all tables + figures
# =========================
run_table_info = plot_run_overview()

task_metric_report = make_detailed_task_metrics(merge_df)
task_metric_report.to_csv(ANALYSIS_DIR / 'detailed_task_metrics.csv', index=False)

# Existing core dashboard blocks.
plot_accuracy_by_label(label_report)
plot_error_count_by_label(label_report)
plot_correct_wrong_stacked(label_report)
plot_missing_rate_by_label(label_report)
plot_accuracy_vs_count(label_report)
plot_output_length_distribution(merge_df)
export_case_review_tables(merge_df, label_report)

# New diagnostics requested after the first dashboard version.
merge_df = add_error_diagnostics(merge_df)
merge_df.to_csv(ANALYSIS_DIR / 'merge_df_with_error_diagnostics.csv', index=False)
merge_df.to_json(ANALYSIS_DIR / 'merge_df_with_error_diagnostics.jsonl', orient='records', lines=True, force_ascii=False)

plot_task_metric_heatmap(task_metric_report)
plot_error_type_distribution(merge_df)
plot_error_type_by_label(merge_df)
plot_extraction_success_by_label(task_metric_report)
plot_output_length_vs_correctness(merge_df)
plot_accuracy_by_difficulty_proxy(merge_df)
plot_answer_format_confusion(merge_df)
write_weakest_case_gallery(merge_df, task_metric_report)
export_wrong_cases_by_label(merge_df)

# Optional blocks: skipped safely unless their input columns/files are present.
plot_version_comparison()
plot_training_data_source_distribution()
plot_inference_cost_dashboard(merge_df)
plot_self_consistency_diagnostics()

if SKIP_LOG:
    skip_info = export_table('99_skipped_optional_diagnostics', pd.DataFrame(SKIP_LOG), max_display_rows=100)
    register_table('Skipped optional diagnostics', skip_info, 'Optional blocks that need additional files or columns.')

dashboard_html_path = make_metrics_dashboard_html(run_table_info)

print('\nGenerated figures:')
for p in sorted(FIGURE_DIR.glob('*.png')):
    print(' -', p)

print('\nGenerated source tables:')
for p in sorted(TABLE_DIR.glob('*.csv')):
    print(' -', p)


## 10. Compact report markdown


In [ ]:
def write_markdown_report():
    """Create a lightweight markdown report that can be copied into a discussion or presentation note."""
    acc = merge_df['is_correct'].mean()
    n = len(merge_df)
    wrong = int((~merge_df['is_correct']).sum())

    if 'label' in merge_df.columns:
        weakest = label_report.head(5)[['label', 'n', 'accuracy', 'wrong']].copy()
        strongest = label_report.sort_values('accuracy', ascending=False).head(5)[['label', 'n', 'accuracy', 'wrong']].copy()
        weakest_md = weakest.to_markdown(index=False)
        strongest_md = strongest.to_markdown(index=False)
    else:
        weakest_md = 'No label column found.'
        strongest_md = 'No label column found.'

    figure_files = '\n'.join(f'- `{p}`' for p in sorted(FIGURE_DIR.glob('*.png')))
    table_files = '\n'.join(f'- `{p}`' for p in sorted(TABLE_DIR.glob('*.csv')))

    text = f"""# Nemotron Local CV Report

## Overall

- Validation cases: {n}
- Correct cases: {n - wrong}
- Wrong cases: {wrong}
- Accuracy: {acc:.6f}
- Figure DPI: {CFG.get('PLOT_DPI', 300)}

## Weakest task types

{weakest_md}

## Strongest task types

{strongest_md}

## Generated 300-DPI figures

{figure_files}

## Matching source tables

{table_files}

## Dashboard

- `analysis_outputs/local_cv_metrics_dashboard.html`
- `analysis_outputs/weakest_label_wrong_cases.html`

## Exported diagnostic files

- `analysis_outputs/per_label_metrics.csv`
- `analysis_outputs/detailed_task_metrics.csv`
- `analysis_outputs/overall_summary.csv`
- `analysis_outputs/wrong_cases.csv`
- `analysis_outputs/correct_cases.csv`
- `analysis_outputs/missing_extraction_cases.csv`
- `analysis_outputs/merge_df.jsonl`
- `analysis_outputs/merge_df_with_error_diagnostics.csv`
- `analysis_outputs/wrong_cases_by_label/`
"""
    report_path = ANALYSIS_DIR / 'local_cv_report.md'
    report_path.write_text(text, encoding='utf-8')
    print('Saved markdown report:', report_path)
    display(Markdown(text))


write_markdown_report()

## 11. Inspect representative good and bad cases


In [ ]:
def print_cases(df: pd.DataFrame, max_cnt: int = 5, title: str = 'Cases'):
    print('\n' + '=' * 120)
    print(title)
    print('=' * 120)
    for i, r in enumerate(df.head(max_cnt).itertuples(index=False)):
        print(f'### CASE {i + 1}')
        if hasattr(r, 'label'):
            print('label:', r.label)
        print('id:', r.id)
        print('answer:', r.answer)
        print('extracted_prediction:', r.extracted_prediction)
        if hasattr(r, 'prompt'):
            print('\n[prompt]\n', str(r.prompt)[:2000])
        print('\n[raw_output]\n', str(r.raw_output)[:3000])
        print('-' * 120)


# Worst categories first, then shortest raw outputs inside each category.
if 'label' in merge_df.columns:
    worst_labels = label_report.head(4)['label'].tolist()
    hard_wrong = wrong_df[wrong_df['label'].isin(worst_labels)].sort_values(['label', 'raw_output_chars'])
else:
    hard_wrong = wrong_df.sort_values('raw_output_chars')

print_cases(correct_df.sample(min(5, len(correct_df)), random_state=CFG['RANDOM_SEED']), title='Random correct cases')
print_cases(hard_wrong, title='Representative wrong cases from weakest labels')


## 12. Create final submission archive


In [ ]:
# Re-create a clean submission.zip from the currently selected adapter.
for required_name in ['adapter_config.json', 'adapter_model.safetensors']:
    if not Path(required_name).exists():
        raise FileNotFoundError(required_name)

with zipfile.ZipFile('submission.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write('adapter_config.json')
    zf.write('adapter_model.safetensors')

print('Created submission.zip')
print('Current output files:')
for p in sorted(Path('.').glob('*')):
    if p.is_file():
        print(' -', p)
print('Figure files:')
for p in sorted(FIGURE_DIR.glob('*')):
    print(' -', p)
print('Analysis files:')
for p in sorted(ANALYSIS_DIR.rglob('*')):
    if p.is_file():
        print(' -', p)
